<a href="https://colab.research.google.com/github/Muhammad-Faiz-Firmansyah/2026_KECERDASAAN_BUATAN/blob/main/jobsheet8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Jobsheet 8 - Data Pre-processing

**Nama:** Muhammad Faiz Firmansyah

**NIM:** 4.33.25.0.16  
**Kelas:** TI-1A

**Dataset:** `iklan_sosmed.csv`

Notebook ini berisi:
1. One Hot Encoding  
2. Outlier Removal  
3. Normalization  
4. Standardization  
5. Train-Test Split  
6. Cross Validation


## Import Library

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.max_columns", None)


## Load Dataset

In [2]:
from google.colab import files

uploaded = files.upload()


Saving iklan_sosmed.csv to iklan_sosmed.csv


In [3]:
csv_name = next(iter(uploaded.keys()))
df = pd.read_csv(csv_name, delimiter=';')

df.head()


,ID,Jenis_Kelamin,Umur,Gaji,Transaksi
0,15624510,Pria,19,285000000,0
1,15810944,Pria,35,300000000,0
2,15668575,Wanita,26,645000000,0
3,15603246,Wanita,27,855000000,0
4,15804002,Pria,19,1140000000,0


In [4]:
print("Shape dataset:", df.shape)
df.info()


Shape dataset: (400, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   ID             400 non-null    int64 
 1   Jenis_Kelamin  400 non-null    object
 2   Umur           400 non-null    int64 
 3   Gaji           400 non-null    int64 
 4   Transaksi      400 non-null    int64 
dtypes: int64(4), object(1)
memory usage: 15.8+ KB


In [5]:
df.isnull().sum()


,0
ID,0
Jenis_Kelamin,0
Umur,0
Gaji,0
Transaksi,0


## 1) Data Preparation dengan One Hot Encoding

In [6]:
# Melihat kategori pada kolom Jenis_Kelamin
df["Jenis_Kelamin"].unique()


array(['Pria', 'Wanita'], dtype=object)

In [7]:
# One Hot Encoding
try:
    encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
except TypeError:
    encoder = OneHotEncoder(sparse=False, handle_unknown="ignore")

encoded_array = encoder.fit_transform(df[["Jenis_Kelamin"]])
encoded_cols = encoder.get_feature_names_out(["Jenis_Kelamin"])

encoded_df = pd.DataFrame(encoded_array, columns=encoded_cols, index=df.index)
df_encoded = pd.concat([df.drop(columns=["Jenis_Kelamin"]), encoded_df], axis=1)

df_encoded.head()


,ID,Umur,Gaji,Transaksi,Jenis_Kelamin_Pria,Jenis_Kelamin_Wanita
0,15624510,19,285000000,0,1.0,0.0
1,15810944,35,300000000,0,1.0,0.0
2,15668575,26,645000000,0,0.0,1.0
3,15603246,27,855000000,0,0.0,1.0
4,15804002,19,1140000000,0,1.0,0.0


## 2) Data Preparation dengan Outlier Removal

In [8]:
# Menghitung z-score untuk kolom Gaji dan Umur
z_score_gaji = np.abs((df_encoded["Gaji"] - df_encoded["Gaji"].mean()) / df_encoded["Gaji"].std())
z_score_umur = np.abs((df_encoded["Umur"] - df_encoded["Umur"].mean()) / df_encoded["Umur"].std())

print("Jumlah outlier pada kolom Gaji :", (z_score_gaji > 3).sum())
print("Jumlah outlier pada kolom Umur :", (z_score_umur > 3).sum())


Jumlah outlier pada kolom Gaji : 0
Jumlah outlier pada kolom Umur : 0


In [9]:
# Menghapus data yang memiliki z-score > 3
mask = (z_score_gaji < 3) & (z_score_umur < 3)
df_clean = df_encoded.loc[mask].copy()

print("Shape sebelum cleaning:", df_encoded.shape)
print("Shape sesudah cleaning :", df_clean.shape)
df_clean.head()


Shape sebelum cleaning: (400, 6)
Shape sesudah cleaning : (400, 6)


,ID,Umur,Gaji,Transaksi,Jenis_Kelamin_Pria,Jenis_Kelamin_Wanita
0,15624510,19,285000000,0,1.0,0.0
1,15810944,35,300000000,0,1.0,0.0
2,15668575,26,645000000,0,0.0,1.0
3,15603246,27,855000000,0,0.0,1.0
4,15804002,19,1140000000,0,1.0,0.0


## 3) Menyiapkan Data untuk Training

In [10]:
# Membagi fitur dan target
X = df_clean.drop(columns=["ID", "Transaksi"])
y = df_clean["Transaksi"]

print("Kolom fitur:", X.columns.tolist())
print("Target:", y.name)


Kolom fitur: ['Umur', 'Gaji', 'Jenis_Kelamin_Pria', 'Jenis_Kelamin_Wanita']
Target: Transaksi


In [11]:
# Split data menjadi training dan testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)


X_train: (320, 4)
X_test : (80, 4)
y_train: (320,)
y_test : (80,)


## 4) Data Preparation dengan Normalization

In [12]:
# Normalisasi menggunakan MinMaxScaler
scaler = MinMaxScaler()

X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

print("Hasil normalisasi 5 data pertama:")
print(X_train_norm[:5])


Hasil normalisasi 5 data pertama:
[[0.14285714 0.31851852 1.         0.        ]
 [0.30952381 0.14074074 0.         1.        ]
 [0.47619048 0.48148148 0.         1.        ]
 [0.95238095 0.05925926 0.         1.        ]
 [0.54761905 0.33333333 0.         1.        ]]


## 5) Data Preparation dengan Standardization

In [13]:
# Standardisasi menggunakan StandardScaler
standardizer = StandardScaler()

X_train_std = standardizer.fit_transform(X_train)
X_test_std = standardizer.transform(X_test)

print("Hasil standardisasi 5 data pertama:")
print(X_train_std[:5])


Hasil standardisasi 5 data pertama:
[[-1.23446186 -0.367992    1.08483674 -1.08483674]
 [-0.57764605 -1.05831408 -0.92179769  0.92179769]
 [ 0.07916976  0.26480323 -0.92179769  0.92179769]
 [ 1.95578636 -1.3747117  -0.92179769  0.92179769]
 [ 0.36066225 -0.31046516 -0.92179769  0.92179769]]


## 6) Cross Validation

In [14]:
# Model klasifikasi sederhana
model = DecisionTreeClassifier(random_state=42)

# 5-fold cross validation
scores = cross_val_score(model, X_train, y_train, cv=5)

print("Hasil cross validation:", scores)
print("Rata-rata akurasi CV:", scores.mean())


Hasil cross validation: [0.90625 0.84375 0.9375  0.875   0.875  ]
Rata-rata akurasi CV: 0.8875


## Kesimpulan

Berdasarkan proses data preparation yang telah dilakukan, dataset `iklan_sosmed.csv` berhasil diproses melalui beberapa tahap penting. Kolom kategorik `Jenis_Kelamin` diubah menjadi data numerik menggunakan One Hot Encoding agar dapat digunakan dalam analisis machine learning. Selanjutnya, data outlier pada kolom `Gaji` dan `Umur` dihapus menggunakan metode z-score supaya data menjadi lebih bersih dan representatif.

Setelah itu, dataset dibagi menjadi data training dan data testing dengan perbandingan 80:20. Data juga diproses menggunakan normalisasi dan standardisasi untuk menyesuaikan skala fitur numerik. Terakhir, dilakukan cross validation untuk melihat kestabilan performa model pada beberapa pembagian data.

Secara keseluruhan, tahap data preparation ini membantu menyiapkan data agar lebih rapi, konsisten, dan siap digunakan untuk pemodelan machine learning.
